# 🏋️ Day 5 실습 — 상태 관리 & LLM 비용 시뮬레이터

📖 **강의 연계**: Day 5 강의자료 「5-1 | 요소 4: 상태 관리」·「5-1 | 요소 6: 비용·지연 계획」·「5-1 | ⭐ 심화 실습」

**✅ 완료 기준**

**[요소 4 · 상태 관리]**
- [ ] LLM의 Stateless 특성과 `MessagesPlaceholder`로 해결하는 방식을 설명할 수 있다
- [ ] `RunnableWithMessageHistory`로 다중 턴 대화가 동작하는 것을 확인했다
- [ ] 팀 서비스에 맞는 Memory 전략을 선택하고 이유를 말할 수 있다
- [ ] 🔰 기본 미션: `day5_실습_팀설계서v1.md` 섹션 4에 상태 관리 전략을 채워 넣었다

**[요소 6 · 비용 계획]**
- [ ] 내 서비스의 요청당 비용과 월 비용을 숫자로 제시할 수 있다
- [ ] 🔰 기본 미션: `day5_실습_팀설계서v1.md` 섹션 7에 비용 수치를 채워 넣었다
- [ ] (⭐ 심화 C) gpt-4o vs gpt-4o-mini 모델 선택 이유를 슬랙 **#day5-심화**에 공유했다

> ⚠️ **시작 전 확인**: 우측 상단 커널이 **`.venv`** 인지 확인하세요 (Colab이 아닌 로컬 VS Code 환경)

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()  # .env 파일에서 환경변수 로드

# tiktoken은 비용 계획 Step 1~3에서 필요
try:
    import tiktoken
    print('✅ tiktoken 준비 완료')
except ImportError:
    assert False, (
        '❌ tiktoken 없음 — 터미널에서 설치하세요: pip install tiktoken'
    )

# langchain 패키지 확인 (상태 관리 Step에서 필요)
try:
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI
    print('✅ langchain 준비 완료')
except ImportError:
    assert False, (
        '❌ langchain 없음 — 터미널에서 설치하세요:\n'
        '   pip install langchain langchain-openai'
    )

# API 키 확인 — 상태 관리 Step과 비용 계획 심화 A/B/C에서 필요
HAS_API_KEY = bool(os.getenv('OPENAI_API_KEY'))
if HAS_API_KEY:
    print(f"✅ OPENAI_API_KEY 확인됨 (앞 7자: {os.getenv('OPENAI_API_KEY')[:7]}...)")
    print(f"  LangSmith 프로젝트: {os.getenv('LANGCHAIN_PROJECT', '(미설정 — LANGCHAIN_PROJECT를 .env에 추가하세요)')}")
else:
    print('ℹ️  OPENAI_API_KEY 없음')
    print('   → 비용 계획 Step 1~3(비용 계산)은 지금 바로 실행 가능합니다')
    print('   → 상태 관리 Step과 심화 A/B/C는 .env에 키를 추가한 후 해당 셀부터 재실행하세요')
    print('   (📖 강의자료 Day 1 모듈 1-4 참조)')

print('\n환경 점검 완료')

✅ tiktoken 준비 완료
✅ langchain 준비 완료
✅ OPENAI_API_KEY 확인됨 (앞 7자: sk-proj...)
  LangSmith 프로젝트: lgcns-agentic-ai

환경 점검 완료


In [3]:
# 이 노트북 전체에서 공유하는 설정 — 맨 처음 한 번만 실행합니다
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o-mini')

# 모델 단가 딕셔너리 (최신가 확인 권장: https://openai.com/api/pricing/)
PRICES = {
    'gpt-4o-mini': {'input': 0.15,  'output': 0.60 },  # $/1M 토큰
    'gpt-4o'     : {'input': 2.50,  'output': 10.00},
}
WON = 1348  # 참고 환율

def calc_cost(in_tok: int, out_tok: int, model: str) -> float:
    '''토큰 수와 모델명으로 요청 1건 비용(달러)을 계산합니다.'''
    p = PRICES[model]
    return (in_tok * p['input'] / 1_000_000) + (out_tok * p['output'] / 1_000_000)

print('✅ 공통 설정 완료 (tiktoken 인코더 · PRICES 딕셔너리 · calc_cost 함수)')

✅ 공통 설정 완료 (tiktoken 인코더 · PRICES 딕셔너리 · calc_cost 함수)


---
## 요소 4. 상태 관리 — LLM의 "건망증" 해결하기

📖 **강의 연계**: Day 5 강의자료 「5-1 | 요소 4: 상태 관리 — LLM의 "건망증" 해결하기」

LLM은 기본적으로 **Stateless(상태 없음)** 입니다. 모델을 호출할 때마다 완전히 새로운 상태에서 시작합니다.
대화를 이어가려면 **이전 대화 내역을 매번 프롬프트에 직접 넣어줘야** 합니다.

| 셀 | 단계 | 목표 |
|----|------|------|
| Step 1-① | 그대로 실행 | LLM 건망증 직접 확인 — 이름을 알려줬는데 모르는 상황 체험 |
| ↳ 참고 | 빌드업 | 이전 대화를 직접 하드코딩하는 방식과 그 한계 이해 |
| Step 1-② | 한 곳만 바꾸기 | `MessagesPlaceholder`로 이전 대화 수동 주입 — 기억 복원 |
| ↳ 참고 | 구조 확인 | `format_messages()`로 LLM에 전달되는 메시지 구조 시각화 |
| Step 1-③ | 내 것에 적용 | `RunnableWithMessageHistory`로 저장·주입·누적 자동화 |
| 🔰 기본 미션 | 내 것에 적용 | 팀 서비스 시나리오 적용 + 설계서 v1 섹션 4 출력 |
| ⭐ 심화 | 확장 | `partial_variables` — 일부 변수를 미리 고정해 재사용성 높이기 |

In [4]:
# ─── 상태 관리 Step 1-① 그대로 실행 — LLM 건망증 확인 ───────────────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | LLM은 왜 건망증이 있는가?」
#
# 이름을 알려줘도 다음 invoke() 호출에서 기억하지 못하는 것을 직접 확인합니다.
# ⚠️ 이 셀은 API 호출이 필요합니다 (약 2회 · $0.001 미만)

if not HAS_API_KEY:
    print("ℹ️ Step 1-①: API 키 없음 → 이 셀은 건너뜁니다")
    print("   .env에 OPENAI_API_KEY를 추가한 뒤 이 셀부터 다시 실행하세요")
    print("   (📖 강의자료 Day 1 모듈 1-4 참조)")
    llm_mem = None  # Run All 통과용
else:
    llm_mem = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # 턴 1: 이름을 알려줌
    r1 = llm_mem.invoke([HumanMessage(content="안녕! 내 이름은 태규야. 기억해줘!")])
    print("턴 1 응답:", r1.content[:100])
    print("-" * 40)

    # 턴 2: 새로운 invoke() — 이전 대화 없음
    r2 = llm_mem.invoke([HumanMessage(content="내 이름이 뭐야?")])
    print("턴 2 응답:", r2.content[:100])
    # 예상 출력: "이름을 알려주시지 않았습니다" 또는 "알 수 없습니다"

    print()
    print("💡 매 invoke()는 완전히 새 상태에서 시작 = Stateless")
    print("   이전 대화를 넣어주지 않으면 100% 기억하지 못합니다.")

턴 1 응답: 안녕, 태규! 반가워! 너와 대화하게 되어 기뻐. 어떤 이야기를 나눌까?
----------------------------------------
턴 2 응답: 죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 당신의 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!

💡 매 invoke()는 완전히 새 상태에서 시작 = Stateless
   이전 대화를 넣어주지 않으면 100% 기억하지 못합니다.


In [5]:
# ─── 빌드업: 대화 이력을 직접 프롬프트에 넣는 방식과 그 한계 ────────────────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | 요소 4: 상태 관리 — LLM의 "건망증" 해결하기」
#
# Step 1-①에서 확인한 건망증을 해결하는 가장 단순한 방법:
# 이전 대화를 ChatPromptTemplate에 그대로 하드코딩합니다.
# → 동작은 하지만, 대화가 길어질수록 코드가 폭발적으로 늘어나는 문제가 있습니다.

if not HAS_API_KEY:
    print("ℹ️ 이 셀은 건너뜁니다 — API 키가 필요합니다")
else:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    # 이전 대화를 '직접' 프롬프트 안에 하드코딩
    prompt_hardcoded = ChatPromptTemplate.from_messages([
        ("system", "너는 한국의 예의바른 교사야. 짧고 구조적으로 대답해줘."),
        ("ai",     "안녕하세요! 무엇을 도와드릴까요?"),
        ("human",  "안녕? 내 이름은 태규야!"),              # ← 이전 대화를 그대로 넣음
        ("ai",     "반가워요, 태규님! 앞으로 잘 부탁드려요."),
        ("human",  "{input}")
    ])

    result = (prompt_hardcoded | llm_mem | StrOutputParser()).invoke(
        {"input": "내 이름이 뭐야?"}
    )
    print("응답:", result)
    # 예상 출력: "태규님이라고 하셨습니다!"

    print()
    print("✅ 기억은 됩니다. 하지만 아래 문제가 있습니다:")
    print("  ❌ 대화가 1턴 → 10턴 → 50턴으로 늘어날 때마다 코드를 수정해야 합니다.") #prompt_hardcoded를 수정해야 한다
    print("  ❌ 여러 사용자의 대화를 따로 관리하기 어렵습니다.")
    print("  ❌ 템플릿이 고정되어 있어 동적 이력 삽입이 불가능합니다.")
    print()
    print("  → 해결책: MessagesPlaceholder (Step 1-②에서 바로 해결합니다)")

응답: 당신의 이름은 태규입니다. 맞나요?

✅ 기억은 됩니다. 하지만 아래 문제가 있습니다:
  ❌ 대화가 1턴 → 10턴 → 50턴으로 늘어날 때마다 코드를 수정해야 합니다.
  ❌ 여러 사용자의 대화를 따로 관리하기 어렵습니다.
  ❌ 템플릿이 고정되어 있어 동적 이력 삽입이 불가능합니다.

  → 해결책: MessagesPlaceholder (Step 1-②에서 바로 해결합니다)


In [6]:
# ─── 상태 관리 Step 1-② 한 곳만 바꾸기 — MessagesPlaceholder로 해결 ──────
# 📖 강의 연계: Day 5 강의자료 「5-1 | 상태 관리 구현 3단계」 ① in-memory 딕셔너리
#
# MessagesPlaceholder: 이전 대화 목록을 통째로 끼워넣는 전용 자리
#   {변수}가 문자열 1개를 받는다면, MessagesPlaceholder는 메시지 리스트를 받습니다.
#
# TODO(🔰): SYSTEM_CONTENT를 내 서비스 시스템 프롬프트로 바꿔보세요.
#   지금은 "예의바른 교사"로 설정되어 있습니다. (그대로도 실행 가능)

if not HAS_API_KEY:
    print("ℹ️ Step 1-②: API 키 없음 → 이 셀은 건너뜁니다")
    chain_with_history = None  # Run All 통과용
else:
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.output_parsers import StrOutputParser

    SYSTEM_CONTENT = "너는 내 비서야. 짧고 구조적으로 대답해줘."
    # TODO(🔰): ↑ 내 서비스 시스템 프롬프트로 바꿔보세요 (힌트: 6요소 체크리스트 1번)

    prompt_with_history = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_CONTENT),
        MessagesPlaceholder("chat_history"),   # ← 이전 대화 목록 삽입 위치
        ("human", "{input}")
    ])
    chain_with_history = prompt_with_history | llm_mem | StrOutputParser()

    # 이전 대화를 직접 넣어줌 (수동 관리)
    history = [
        ("human", "내 이름은 태규야. 기억해줘"),
        ("ai",    "안녕하세요, 태규님! 잘 기억하겠습니다."),
    ]

    result = chain_with_history.invoke({
        "input": "내 이름이 뭐야?",
        "chat_history": history    # ← 이전 대화를 직접 전달
    })
    print("응답:", result)
    # 예상 출력: "민재님이라고 하셨습니다!"

    print()
    print("💡 history 리스트에 이전 대화를 넣으면 기억합니다.")
    print("   하지만 매번 직접 관리해야 하는 번거로움이 있습니다.")
    print("   → 아래 셀에서 메시지 구조를 확인한 뒤, Step 1-③에서 자동화합니다.")

응답: 태규님입니다.

💡 history 리스트에 이전 대화를 넣으면 기억합니다.
   하지만 매번 직접 관리해야 하는 번거로움이 있습니다.
   → 아래 셀에서 메시지 구조를 확인한 뒤, Step 1-③에서 자동화합니다.


In [7]:
# ─── Step 1-② 보충 — format_messages()로 LLM 입력 메시지 구조 시각화 ──────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | 상태 관리 구현 3단계」 ① in-memory → ② LangChain 방식
#
# MessagesPlaceholder가 실제로 어떤 메시지를 만들어 LLM에 전달하는지 확인합니다.
# format_messages()를 호출하면 LLM이 받을 최종 메시지 목록을 미리 볼 수 있습니다.
# → "블랙박스를 열어보는 것" — 구조를 이해하면 디버깅이 쉬워집니다.

if not HAS_API_KEY:
    print("ℹ️ 이 셀은 건너뜁니다 — API 키가 필요합니다")
elif chain_with_history is None:
    print("ℹ️ Step 1-②가 실행되지 않았습니다 — 먼저 Step 1-② 셀을 실행하세요")
else:
    print("=== MessagesPlaceholder가 채워진 최종 메시지 구조 ===")
    print()
    messages = prompt_with_history.format_messages(
        input="내 이름이 뭐야?",
        chat_history=history   # Step 1-②에서 정의한 history 변수
    )

    for i, msg in enumerate(messages, 1):
        msg_type = type(msg).__name__
        preview = msg.content[:55] + ("..." if len(msg.content) > 55 else "")
        print(f"  [{i}] {msg_type:<25} | {preview}")

    print()
    print("💡 LLM이 실제로 받는 순서:")
    print("   SystemMessage → HumanMessage(이전) → AIMessage(이전) → HumanMessage(현재)")
    print()
    print("   MessagesPlaceholder('chat_history')는 chat_history 리스트를")
    print("   SystemMessage와 HumanMessage(현재) 사이에 그대로 끼워넣습니다.")
    print()
    print("   {input}     ← 문자열 1개를 받는 일반 변수")
    print("   Placeholder ← 메시지 리스트 전체를 받는 특수 변수")

=== MessagesPlaceholder가 채워진 최종 메시지 구조 ===

  [1] SystemMessage             | 너는 내 비서야. 짧고 구조적으로 대답해줘.
  [2] HumanMessage              | 내 이름은 태규야. 기억해줘
  [3] AIMessage                 | 안녕하세요, 태규님! 잘 기억하겠습니다.
  [4] HumanMessage              | 내 이름이 뭐야?

💡 LLM이 실제로 받는 순서:
   SystemMessage → HumanMessage(이전) → AIMessage(이전) → HumanMessage(현재)

   MessagesPlaceholder('chat_history')는 chat_history 리스트를
   SystemMessage와 HumanMessage(현재) 사이에 그대로 끼워넣습니다.

   {input}     ← 문자열 1개를 받는 일반 변수
   Placeholder ← 메시지 리스트 전체를 받는 특수 변수


In [12]:
# ─── 상태 관리 Step 1-③ 내 것에 적용 — RunnableWithMessageHistory 자동화 ──
# 📖 강의 연계: Day 5 강의자료 「5-1 | 상태 관리 구현 3단계」 ② LangChain 방식
#
# RunnableWithMessageHistory가 자동으로:
#   1) get_history(session_id)로 과거 이력 로드
#   2) MessagesPlaceholder 자리에 삽입
#   3) 실행 후 이번 대화(human + ai)를 스토어에 저장
#
# session_id를 사용자마다 다르게 설정하면 여러 사용자의 대화가 독립 관리됩니다.

if not HAS_API_KEY:
    print("ℹ️ Step 1-③: API 키 없음 → 이 셀은 건너뜁니다")
    chat  = None   # Run All 통과용
    DEMO_CFG = {"configurable": {"session_id": "demo"}}
else:
    from langchain_core.runnables.history import RunnableWithMessageHistory
    from langchain_core.chat_history import InMemoryChatMessageHistory

    memory_store = {}   # session_id → InMemoryChatMessageHistory

    def get_history(session_id: str):
        # key가 없으면 새로 만들고, 있으면 기존 것 반환
        return memory_store.setdefault(session_id, InMemoryChatMessageHistory())

    def clear_history(session_id: str):
        memory_store.setdefault(session_id, InMemoryChatMessageHistory()).clear()

    chat = RunnableWithMessageHistory(
        chain_with_history,
        get_history,
        input_messages_key="input",          # 프롬프트의 human 변수명과 일치해야 함
        history_messages_key="chat_history"  # MessagesPlaceholder 변수명과 일치해야 함
    )

    DEMO_CFG = {"configurable": {"session_id": "user_태규"}}

    print("=== 자동 메모리 테스트 ===")
    r1 = chat.invoke({"input": "내 이름은 태규야. 나는 노래맞추기 맵 만들기 전문가야. 노래맞추기 맵은 스타크래프트를 활용해서 만들고 있어."}, DEMO_CFG)
    print(f"대화 1: {r1[:80]}")

    r2 = chat.invoke({"input": "방금 무슨 얘기 했는지 한 문장으로 요약해줘."}, DEMO_CFG)
    print(f"대화 2: {r2[:80]}")

    r3 = chat.invoke({"input": "내 이름이 뭐야?"}, DEMO_CFG)
    print(f"대화 3: {r3[:80]}")    # "민재님이라고 하셨습니다!"

    # print()
    # clear_history(DEMO_CFG["configurable"]["session_id"])
    # r4 = chat.invoke({"input": "내 이름이 뭐야?"}, DEMO_CFG)
    # print(f"(초기화 후) 대화 4: {r4[:80]}")   # ← 이름을 모름

    print()
    print("💡 실무에서는 memory_store(dict) 대신 DB(Redis·PostgreSQL)를 사용합니다.")
    print("   이후 파이프라인 모듈: SQLChatMessageHistory로 교체하면 서버 재시작 후에도 유지됩니다.")

c:\Users\Admin\Desktop\실습용\my_llm_service\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


=== 자동 메모리 테스트 ===
대화 1: 안녕하세요, 태규님! 노래맞추기 맵 제작에 대해 어떤 도움이 필요하신가요? 아이디어, 기술적 조언, 또는 특정 기능에 대한 질문이 있으신가요?
대화 2: 태규님은 스타크래프트를 활용해 노래맞추기 맵 만들기 전문가입니다.
대화 3: 태규님입니다.

💡 실무에서는 memory_store(dict) 대신 DB(Redis·PostgreSQL)를 사용합니다.
   이후 파이프라인 모듈: SQLChatMessageHistory로 교체하면 서버 재시작 후에도 유지됩니다.


In [13]:
memory_store

{'user_태규': InMemoryChatMessageHistory(messages=[HumanMessage(content='내 이름은 태규야. 나는 노래맞추기 맵 만들기 전문가야. 노래맞추기 맵은 스타크래프트를 활용해서 만들고 있어.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 태규님! 노래맞추기 맵 제작에 대해 어떤 도움이 필요하신가요? 아이디어, 기술적 조언, 또는 특정 기능에 대한 질문이 있으신가요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='방금 무슨 얘기 했는지 한 문장으로 요약해줘.', additional_kwargs={}, response_metadata={}), AIMessage(content='태규님은 스타크래프트를 활용해 노래맞추기 맵 만들기 전문가입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}), AIMessage(content='태규님입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

## 🔰 상태 관리 기본 미션 — 팀 서비스에 Memory 전략 적용

📖 **강의 연계**: Day 5 강의자료 「5-1 | Memory 전략 선택 기준」·「5-3② 설계서 v1 작성」

Step 1-③에서 완성한 `chat` 체인을 팀 서비스 시나리오에 적용하고,  
결과를 `day5_실습_팀설계서v1.md` **섹션 4(상태 관리)** 에 기재합니다.

| 단계 | 내용 |
|------|------|
| 변수 채우기 | `MY_SERVICE_NAME`, `MY_MEMORY_STRATEGY`, `turns` 리스트 수정 |
| 자동 실행 | 팀 서비스 시나리오로 다중 턴 대화 확인 |
| 출력 복사 | 설계서 v1 섹션 4 삽입용 텍스트 출력 → 복사해서 붙여넣기 |

> ⚠️ Step 1-③ 셀이 실행되어 있어야 `chat` 변수가 존재합니다.

In [14]:
# 🔰 상태 관리 기본 미션 — 팀 서비스에 Memory 적용 + 설계서 v1 섹션 4 출력
# Step 1-③의 chat 객체가 먼저 실행되어 있어야 합니다.

# ─── 1단계: 팀 서비스 설정 ───────────────────────────────────────────────────
MY_SERVICE_NAME    = "회의록 요약 서비스"      # ← 팀 서비스명으로 바꾸세요
MY_MEMORY_STRATEGY = "Sliding Window"         # ← Sliding Window / Summary / Entity / Vector 중 택1
MY_WINDOW_SIZE     = 5                        # ← Sliding Window 선택 시: 유지할 턴 수
MY_SESSION_NEEDED  = "Y"                      # ← 세션 유지 필요 여부 Y / N
# (그대로도 실행됩니다 — 단, 팀 서비스에 맞게 바꿔야 설계서 활용 가능)

# ─── 2단계: 팀 서비스 시나리오로 다중 턴 대화 테스트 ──────────────────────────
if HAS_API_KEY and chat is not None:
    SERVICE_CFG = {"configurable": {"session_id": "team_service_demo"}}
    # TODO(🔰): 아래 turns를 팀 서비스의 실제 사용 시나리오로 교체하세요
    turns = [
        "안녕하세요! 오늘 팀 미팅 회의록을 정리해 주세요.",
        "참석자는 5명이었고 Q3 예산 증액을 주로 논의했어요.",
        "방금 말한 논의 주제를 다시 알려줄 수 있나요?",  # ← 이전 대화 기억 확인
    ]
    print("=== 팀 서비스 Memory 테스트 ===")
    for i, turn in enumerate(turns, 1):
        resp = chat.invoke({"input": turn}, SERVICE_CFG)
        print(f"사용자 [{i}]: {turn[:60]}")
        print(f"  AI 응답: {resp[:120]}")
        print()
else:
    print("ℹ️ API 키 없음 — 시나리오 테스트 건너뜀 (전략 선택 출력만 진행)")

# ─── 3단계: 설계서 v1 섹션 4 삽입용 출력 ─────────────────────────────────────
REASONS = {
    "Sliding Window": "짧은 Q&A 위주, 오래된 이력 불필요, 비용 안정",
    "Summary":        "긴 상담/전체 흐름 중요, 세부보다 맥락이 핵심",
    "Entity":         "사용자 이름·날짜 등 명시적 사실 정확 보존이 중요",
    "Vector":         "장기 이력 검색 필요, RAG 기반 챗봇 구조",
}
print("=" * 60)
print("📋 day5_실습_팀설계서v1.md 섹션 4 삽입용 요약")
print("   아래를 복사해 설계서 '## 4. 상태 관리' 항목에 붙여넣으세요")
print("=" * 60)
print()
print(f"**서비스명**: {MY_SERVICE_NAME}")
print(f"**선택 전략**: {MY_MEMORY_STRATEGY}")
if MY_MEMORY_STRATEGY == "Sliding Window":
    print(f"**유지 턴 수**: 최근 {MY_WINDOW_SIZE}턴")
print(f"**세션 유지 필요**: {MY_SESSION_NEEDED}")
print(f"**선택 이유**: {REASONS.get(MY_MEMORY_STRATEGY, '직접 작성')}")
print(f"**현재 저장소**: InMemoryChatMessageHistory (서버 재시작 시 초기화)")
print(f"**이후 파이프라인 모듈**: SQLChatMessageHistory (PostgreSQL 영속화 예정)")
print()
print("💡 MY_MEMORY_STRATEGY를 바꾸면 선택 이유 문구도 자동으로 바뀝니다.")

=== 팀 서비스 Memory 테스트 ===
사용자 [1]: 안녕하세요! 오늘 팀 미팅 회의록을 정리해 주세요.
  AI 응답: 안녕하세요! 팀 미팅 회의록 정리해 드리겠습니다. 다음과 같은 형식으로 작성할 수 있습니다:

---

**팀 미팅 회의록**

**일시:** [날짜 및 시간]  
**장소:** [회의 장소]  
**참석자:** [

사용자 [2]: 참석자는 5명이었고 Q3 예산 증액을 주로 논의했어요.
  AI 응답: 알겠습니다! 다음과 같이 회의록을 정리해 드리겠습니다.

---

**팀 미팅 회의록**

**일시:** [날짜 및 시간]  
**장소:** [회의 장소]  
**참석자:** 5명  

**1. 안건 1:** Q3 

사용자 [3]: 방금 말한 논의 주제를 다시 알려줄 수 있나요?
  AI 응답: 논의 주제는 **Q3 예산 증액**입니다. 이 주제에 대해 예산 증액의 필요성과 각 부서의 요구 사항을 검토하는 내용이 논의되었습니다. 추가 질문이 있으시면 말씀해 주세요!

📋 day5_실습_팀설계서v1.md 섹션 4 삽입용 요약
   아래를 복사해 설계서 '## 4. 상태 관리' 항목에 붙여넣으세요

**서비스명**: 회의록 요약 서비스
**선택 전략**: Sliding Window
**유지 턴 수**: 최근 5턴
**세션 유지 필요**: Y
**선택 이유**: 짧은 Q&A 위주, 오래된 이력 불필요, 비용 안정
**현재 저장소**: InMemoryChatMessageHistory (서버 재시작 시 초기화)
**이후 파이프라인 모듈**: SQLChatMessageHistory (PostgreSQL 영속화 예정)

💡 MY_MEMORY_STRATEGY를 바꾸면 선택 이유 문구도 자동으로 바뀝니다.


## ⭐ 심화 — `partial_variables`: 일부 변수를 미리 고정하기

📖 **강의 연계**: Day 5 강의자료 「5-1 | 6요소 설계 결정」 프롬프트 전략

`partial_variables`는 템플릿 변수 중 일부를 미리 고정하고 나머지만 동적으로 받는 방법입니다.

| 항목 | 내용 |
|------|------|
| **활용 상황** | 역할(R)·도메인(C)은 서비스에서 고정, 질문(I)·형식(F)만 동적으로 변경 |
| **팀 서비스 예시** | `domain='회계 분야'`를 고정 → `{task}`만 바꿔 다양한 질문에 재사용 |
| **partial 후** | 고정된 변수는 `partial_variables`로 이동, `input_variables`에서 제거 |

In [ ]:
# ─── ⭐ 심화: partial_variables — 일부 변수를 미리 고정하기 ──────────────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | 6요소 설계 결정」 프롬프트 전략
#
# partial(): 변수 일부를 미리 고정 → 나머지만 invoke 시 전달하면 됩니다.
# 팀 서비스에서 '역할(R)'은 고정하고 '질문 내용(I)'만 동적으로 받을 때 유용합니다.

if not HAS_API_KEY:
    print("ℹ️ ⭐ 심화: API 키 없음 → 이 셀은 건너뜁니다")
else:
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import StrOutputParser  # 이 셀에서 직접 사용

    # ─── 두 변수({word}, {meaning})를 가진 기본 템플릿 ───────────────────────────
    template = "{word}의 {meaning}은 무엇인가요? 단어만 대답해 주세요."
    prompt_two = PromptTemplate(
        template=template,
        input_variables=["word", "meaning"],
    )
    print("원본 input_variables:", prompt_two.input_variables)
    # 출력: ['word', 'meaning'] — 두 변수 모두 채워야 실행 가능

    print()

    # ─── partial(): 'meaning'을 "반대말"로 미리 고정 ──────────────────────────
    partial_pmt = prompt_two.partial(meaning="반대말")
    print("partial 후 input_variables:", partial_pmt.input_variables)
    print("partial 후 partial_variables:", partial_pmt.partial_variables)
    # 출력: input_variables=['word'] / partial_variables={'meaning': '반대말'}

    print()

    chain_partial = partial_pmt | llm_mem | StrOutputParser()

    # word만 넣어도 실행 가능 (meaning은 자동으로 "반대말" 사용)
    r1 = chain_partial.invoke({"word": "사랑"})
    print(f"word=사랑, meaning=반대말(고정): {r1}")

    # invoke 시점에 meaning을 다시 넣으면 덮어쓰기 가능
    r2 = chain_partial.invoke({"word": "사랑", "meaning": "비슷한 말"})
    print(f"word=사랑, meaning=비슷한 말(덮어쓰기): {r2}")

    print()
    print("─" * 50)
    print("💡 팀 서비스 적용 아이디어:")
    print()

    # TODO(⭐): 아래 예시를 팀 서비스의 실제 프롬프트로 바꿔보세요.
    from langchain_core.prompts import ChatPromptTemplate

    team_prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 {domain} 전문가입니다. {tone} 어조로 한국어로 답하세요."),
        ("human",  "{task}")
    ])

    # domain과 tone을 팀 서비스에 맞게 미리 고정
    partial_team = team_prompt.partial(
        domain="비즈니스 커뮤니케이션",  # ← 팀 서비스 도메인으로 교체
        tone="간결하고 전문적인"         # ← 팀 서비스 어조로 교체
    )

    chain_team = partial_team | llm_mem | StrOutputParser()

    # 이제 {task}만 바꾸면 됩니다
    tasks = [
        "이메일 제목을 5자 이내로 요약해줘: '다음 주 월요일 오후 프로젝트 킥오프 미팅 참석 확인 건'",
        "회의 안건을 우선순위 순서로 3개만 뽑아줘: 예산 검토, 일정 조율, 팀 빌딩, 기술 스택 결정",
    ]

    for task in tasks:
        result = chain_team.invoke({"task": task})
        print(f"Q: {task[:45]}...")
        print(f"A: {result[:80]}")
        print()

---
## 요소 6. 비용·지연 계획 — LLM 비용 시뮬레이터

📖 **강의 연계**: Day 5 강의자료 「5-1 | 요소 6: 비용·지연 계획」·「5-1 | ⭐ 심화 실습」

아래 Step 1~3에서 계산한 비용 수치를 설계서 v1 **섹션 7(비기능 요구사항)** 에 기재합니다.

---
## Step 1. 토큰이란? — tiktoken으로 실측하기

📖 **강의 연계**: Day 5 강의자료 「5-1 | 요소 6: 비용·지연 계획」 비용·지연 계획 표

LLM 비용은 **토큰 수 × 단가**입니다. 토큰은 단어보다 작은 단위입니다.
- 한국어 1글자 ≈ 1.5~2 토큰 (영어보다 비쌈)
- `"안녕하세요"` = 4토큰 / `"Hello"` = 1토큰

| 셀 | 목표 |
|----|------|
| Step 1-① | 예시 텍스트로 토큰 수를 실측 — 한국어·영어 차이 체감 |
| Step 1-② | **내 서비스 프롬프트로 교체** — 실제 입력 토큰 측정 |

In [ ]:
# ─── Step 1-① 그대로 실행 — 예시 텍스트 토큰 수 비교 ──────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | 요소 6: 비용·지연 계획」 비용·지연 계획 표

examples = {
    '짧은 한국어'      : '안녕하세요',
    '짧은 영어'        : 'Hello',
    '시스템 프롬프트 예시': (
        '당신은 비즈니스 커뮤니케이션 전문가입니다. '
        '회의 원문을 받으면 핵심 결정 사항, 액션 아이템, '
        '다음 회의 안건을 JSON 형식으로 반환하세요.'
    ),
    '사용자 입력 예시'  : (
        '오늘 회의에서 Q3 마케팅 예산을 20% 늘리기로 했습니다. '
        '김팀장은 다음 주까지 집행 계획서를 제출하기로 했습니다.'
    ),
}

print('텍스트별 토큰 수 측정')
print('-' * 52)
for name, text in examples.items():
    t = len(enc.encode(text))
    print(f'{name:<20}: {t:>4}토큰  (글자수: {len(text)})')
# 예상 출력:
# 짧은 한국어         :    4토큰  (글자수: 5)
# 짧은 영어           :    1토큰  (글자수: 5)
# 시스템 프롬프트 예시 :   약 52토큰
# 사용자 입력 예시     :   약 53토큰

In [ ]:
# ─── Step 1-② 한 곳만 바꾸기 — 내 프롬프트 토큰 측정 ──────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | ⭐ 심화 실습」 + 오늘 작성한 6요소 체크리스트
#
# TODO(🔰): MY_SYSTEM_PROMPT와 MY_SAMPLE_INPUT을 내 서비스 프롬프트로 바꾸세요.
#   힌트: day5_실습_개인체크리스트.md Part 1에 작성한 내용을 붙여넣으면 됩니다.
#   힌트2: 시스템 프롬프트가 길수록 매 요청 비용이 증가합니다 — 핵심만 남기세요.

MY_SYSTEM_PROMPT = '''당신은 비즈니스 커뮤니케이션 전문가입니다.
사용자가 회의 원문을 제공하면 핵심 결정 사항, 액션 아이템,
다음 회의 안건을 JSON 형식으로 반환하세요.'''  # ← 내 시스템 프롬프트로 교체하세요 (그대로도 실행은 됩니다)

MY_SAMPLE_INPUT = '오늘 회의에서 Q3 예산을 논의했습니다. 마케팅 20% 증액으로 결론났습니다.'  # ← 전형적인 사용자 입력으로 교체하세요 (그대로도 실행은 됩니다)

# 측정 — total_input은 이후 Step 2·3 셀에서 자동으로 참조합니다
system_tokens = len(enc.encode(MY_SYSTEM_PROMPT))
input_tokens  = len(enc.encode(MY_SAMPLE_INPUT))
total_input   = system_tokens + input_tokens

print(f'시스템 프롬프트 : {system_tokens:>4}토큰')
print(f'사용자 입력     : {input_tokens:>4}토큰')
print(f'총 입력 토큰    : {total_input:>4}토큰  ← Step 2·3에서 비용 계산에 사용됩니다')
print()
print('💡 시스템 프롬프트가 길수록 매 요청 비용이 증가합니다.')
print('   핵심 지시만 남기고 나머지를 줄이는 것이 비용 최적화의 첫 단계입니다.')

---
## Step 2. 요청당 비용 계산

📖 **강의 연계**: Day 5 강의자료 비용·지연 계획 표 (`$0.000405/요청` 예시)

**비용 공식**: `비용 = (입력 토큰 × 입력 단가/1M) + (출력 토큰 × 출력 단가/1M)`

| 모델 | 입력 단가(/1M토큰) | 출력 단가(/1M토큰) |
|------|-------------------|-------------------|
| gpt-4o-mini | $0.15 | $0.60 |
| gpt-4o | $2.50 | $10.00 |

> ⚠️ 위 단가는 참고용입니다. 최신가는 https://openai.com/api/pricing/ 에서 확인하세요.

| 셀 | 목표 |
|----|------|
| Step 2-① | 예시 서비스(회의록 요약) 요청당 비용 계산 |
| Step 2-② | **내 서비스 출력 토큰으로 교체** — 모델별 비교 |

In [ ]:
# ─── Step 2-① 그대로 실행 — 예시 서비스 요청당 비용 ───────────
# 📖 강의 연계: Day 5 강의자료 비용·지연 계획 표

# 예시: 회의록 요약 서비스 (시스템 프롬프트 500 + 회의록 1,543 ≈ 2,043 입력 토큰)
ex_input  = 2043
ex_output =  400   # 예상 출력 토큰

print('예시 서비스(회의록 요약) 요청당 비용')
print('-' * 45)
for m in PRICES:
    c = calc_cost(ex_input, ex_output, m)
    print(f'{m:<15}: ${c:.6f} = {c * WON:.2f}원/요청')
# 예상 출력:
# gpt-4o-mini : $0.000547 = 0.74원/요청
# gpt-4o      : $0.009108 = 12.30원/요청

In [ ]:
# ─── Step 2-② 한 곳만 바꾸기 — 내 서비스 출력 토큰 교체 ────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | ⭐ 심화 실습」
#
# TODO(🔰): MY_OUTPUT_TOKENS를 내 서비스 예상 출력 토큰 수로 수정하세요.
#   힌트: 짧은 분류·태깅=50, 중간 요약=200, 긴 분석 보고=500
#   힌트2: MY_INPUT_TOKENS는 Step 1-②의 total_input을 자동으로 가져옵니다.

MY_INPUT_TOKENS  = total_input  # Step 1-②에서 자동 연결 (수정 불필요)
MY_OUTPUT_TOKENS = 500          # ← 여기를 수정하세요 (그대로도 실행은 됩니다)

print(f'내 서비스 요청당 비용 (입력 {MY_INPUT_TOKENS}토큰 + 출력 {MY_OUTPUT_TOKENS}토큰)')
print('-' * 52)
for m in PRICES:
    c = calc_cost(MY_INPUT_TOKENS, MY_OUTPUT_TOKENS, m)
    print(f'{m:<15}: ${c:.6f} = {c * WON:.2f}원/요청')

---
## Step 3. 월 비용 시뮬레이션

📖 **강의 연계**: Day 5 강의자료 비용·지연 계획 표 (월 1만 요청 $4.05 예시)

서비스 사용량이 늘어나면 비용도 선형으로 증가합니다.  
런칭 전에 낙관·비관 시나리오를 모두 계산해두면 예산 충격을 피할 수 있습니다.

| 셀 | 목표 |
|----|------|
| Step 3-① | 예시 서비스 월별 시나리오 표 실행 |
| Step 3-② | **내 서비스 목표 요청 수 입력** — 설계서 v1 수치 확정 |

In [ ]:
# ─── Step 3-① 그대로 실행 — 예시 서비스 월별 시나리오 표 ────────
# 📖 강의 연계: Day 5 강의자료 비용·지연 계획 표

ex_cost = calc_cost(ex_input, ex_output, 'gpt-4o-mini')
volumes = [1_000, 5_000, 10_000, 50_000, 100_000]

print('월 요청 수별 비용 (gpt-4o-mini / 예시: 회의록 요약)')
print('-' * 52)
for v in volumes:
    m = ex_cost * v
    print(f'  {v:>7,}건/월: ${m:>8.2f} = {m * WON:>8,.0f}원')
# 예상 출력:
#    1,000건/월: $    0.55 =      742원
#    5,000건/월: $    2.73 =    3,690원
#   10,000건/월: $    5.47 =    7,381원
#   50,000건/월: $   27.33 =   36,901원
#  100,000건/월: $   54.66 =   73,793원

In [ ]:
# ─── Step 3-② 한 곳만 바꾸기 — 내 서비스 목표 요청 수 입력 ───────
# 📖 강의 연계: Day 5 강의자료 「5-1 | ⭐ 심화 실습」
#
# TODO(🔰): MY_MONTHLY_TARGET을 내 서비스 목표 월 요청 수로 바꾸세요.
#   힌트: 소규모 사내 봇=500, 팀 내부 도구=1,000, 공개 서비스=10,000+

MY_MONTHLY_TARGET = 5_000   # ← 목표 월 요청 수로 수정하세요 (그대로도 실행은 됩니다)

my_cost_per_req = calc_cost(MY_INPUT_TOKENS, MY_OUTPUT_TOKENS, 'gpt-4o-mini')

print('내 서비스 월 비용 시뮬레이션 (gpt-4o-mini)')
print('-' * 52)
for v in [MY_MONTHLY_TARGET // 5, MY_MONTHLY_TARGET, MY_MONTHLY_TARGET * 2]:
    if v > 0:
        m = my_cost_per_req * v
        marker = '  ← 목표' if v == MY_MONTHLY_TARGET else ''
        print(f'  {v:>7,}건/월: ${m:>8.2f} = {m * WON:>8,.0f}원{marker}')

---
## 🔰 기본 미션 — 설계서 v1 섹션 7 삽입용 요약 출력

📖 **강의 연계**: Day 5 강의자료 「5-3② 설계서 v1 작성」·`day5_실습_팀설계서v1.md` 섹션 7

Step 1~3 결과가 자동으로 집계됩니다. 아래 셀을 실행한 뒤  
출력 내용을 `day5_실습_팀설계서v1.md` **섹션 7(비기능 요구사항)** 표에 붙여넣으세요.

| 단계 | 내용 |
|------|------|
| Step 1-② | 토큰 수 실측 → `total_input` 확정 |
| Step 2-② | 출력 토큰 추정 → 요청당 비용 계산 |
| Step 3-② | 목표 월 요청 수 입력 → 월 비용 계산 |
| 🔰 기본 미션 | 위 세 값 집계 → 설계서 v1 삽입용 요약 출력 |

> ⚠️ Step 1-②·2-②·3-② 셀이 먼저 실행되어 있어야 올바른 값이 출력됩니다.

In [ ]:
# 🔰 기본 미션 — Step 1~3 결과를 집계해 설계서 v1 삽입용 요약 출력
# Step 2-②, 3-② 셀 실행 후 이 셀을 실행하세요.

monthly_cost = my_cost_per_req * MY_MONTHLY_TARGET

print('=' * 60)
print('📋 day5_실습_팀설계서v1.md 섹션 7 삽입용 요약')
print('   아래를 복사해 설계서 비기능 요구사항 표에 붙여넣으세요')
print('=' * 60)
print()
print(f'| 항목               | 목표값                     | 근거              |')
print(f'|--------------------|---------------------------|-------------------|')
print(f'| 선택 모델           | gpt-4o-mini               | 비용 효율 우선     |')
print(f'| 입력 토큰 (예상)    | {MY_INPUT_TOKENS}토큰                   |                   |')
print(f'| 출력 토큰 (예상)    | {MY_OUTPUT_TOKENS}토큰                   |                   |')
print(f'| 요청당 비용         | ${my_cost_per_req:.6f}      | 위 토큰 × 단가    |')
print(f'| 월 {MY_MONTHLY_TARGET:,}건 비용    | ${monthly_cost:.2f} ≈ {monthly_cost * WON:,.0f}원 |                   |')
print(f'| 목표 응답 시간      | P50 < 2초, P99 < 5초       | (팀 합의 후 수정)  |')
print()
print('⚠️ 위 단가는 참고용입니다. 최신가: https://openai.com/api/pricing/')
# → 출력을 복사해 day5_실습_팀설계서v1.md 섹션 7에 붙여넣으세요!

---
## ⭐ 심화 미션 — 조기 완료자용

기본 미션 완료 후 아래 **A → B → C 순서**로 진행하세요.

**⭐ 심화 A.** gpt-4o-mini로 내 프롬프트 실제 호출  
**⭐ 심화 B.** gpt-4o로 동일 호출 — 응답 품질·속도 비교  
**⭐ 심화 C.** 두 모델 비교 요약 → 모델 선택 이유 작성

📖 **강의 연계**: Day 5 강의자료 「5-1 | ⭐ 심화 실습 ②: 두 모델 비교 분석」

> ⚠️ 이 셀들은 OpenAI API를 실제로 호출합니다 (비용 발생).  
> API 키가 없으면 셀이 자동으로 건너뜁니다 — 셀 2 환경 점검 결과 참조.

In [ ]:
# ─── ⭐ 심화 A — gpt-4o-mini 실제 호출 ─────────────────────
# 📖 강의 연계: Day 5 강의자료 「5-1 | ⭐ 심화 실습 ②」

import time

if not HAS_API_KEY:
    print('⭐ 심화 A 건너뜀 — OPENAI_API_KEY 없음')
    print('   .env 파일에 키를 추가한 후 이 셀부터 다시 실행하세요.')
    print('   (📖 강의자료 Day 1 모듈 1-4 참조)')
    # Run All 통과를 위한 폴백 변수
    elapsed_mini = 0.0
    tokens_mini  = {'input_tokens': MY_INPUT_TOKENS, 'output_tokens': MY_OUTPUT_TOKENS}
    content_mini = '[API 키 없음 — 실제 응답 없음]'
else:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import SystemMessage, HumanMessage

    llm_mini = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    start = time.time()
    resp = llm_mini.invoke([
        SystemMessage(content=MY_SYSTEM_PROMPT),
        HumanMessage(content=MY_SAMPLE_INPUT),
    ])
    elapsed_mini = time.time() - start
    tokens_mini  = resp.usage_metadata or {}
    content_mini = resp.content

    print('=== gpt-4o-mini 응답 ===')
    print(content_mini[:600])
    print()
    print(f'지연 시간 : {elapsed_mini:.2f}초')
    print(f'입력 토큰 : {tokens_mini.get("input_tokens", "?")}')
    print(f'출력 토큰 : {tokens_mini.get("output_tokens", "?")}')
# → 응답 품질을 확인하세요. 심화 B와 어떻게 다른가요?

In [ ]:
# ─── ⭐ 심화 B — gpt-4o 실제 호출 ─────────────────────────
# ⚠️ gpt-4o는 gpt-4o-mini 대비 약 17배 비쌉니다 — 1회 테스트만 실행하세요.

if not HAS_API_KEY:
    print('⭐ 심화 B 건너뜀 — OPENAI_API_KEY 없음')
    elapsed_full = 0.0
    tokens_full  = {'input_tokens': MY_INPUT_TOKENS, 'output_tokens': MY_OUTPUT_TOKENS}
    content_full = '[API 키 없음 — 실제 응답 없음]'
else:
    llm_full = ChatOpenAI(model='gpt-4o', temperature=0)
    start = time.time()
    resp = llm_full.invoke([
        SystemMessage(content=MY_SYSTEM_PROMPT),
        HumanMessage(content=MY_SAMPLE_INPUT),
    ])
    elapsed_full = time.time() - start
    tokens_full  = resp.usage_metadata or {}
    content_full = resp.content

    print('=== gpt-4o 응답 ===')
    print(content_full[:600])
    print()
    print(f'지연 시간 : {elapsed_full:.2f}초')
    print(f'입력 토큰 : {tokens_full.get("input_tokens", "?")}')
    print(f'출력 토큰 : {tokens_full.get("output_tokens", "?")}')
# → 심화 A와 비교: 응답 품질·길이·구체성에서 어떤 차이가 있나요?

In [ ]:
# ─── ⭐ 심화 C — 두 모델 비교 요약 + 모델 선택 판단 ──────────

in_tok   = tokens_mini.get('input_tokens',  MY_INPUT_TOKENS)
out_mini = tokens_mini.get('output_tokens', MY_OUTPUT_TOKENS)
out_full = tokens_full.get('output_tokens', MY_OUTPUT_TOKENS)

c_mini = calc_cost(in_tok, out_mini, 'gpt-4o-mini')
c_full = calc_cost(in_tok, out_full, 'gpt-4o')
saving = (c_full - c_mini) * MY_MONTHLY_TARGET

print('=' * 58)
print('📊 두 모델 비교 요약')
print('=' * 58)
print(f'{"항목":<16} {"gpt-4o-mini":>18} {"gpt-4o":>16}')
print('-' * 58)
print(f'{"응답 시간":<16} {elapsed_mini:>16.2f}초 {elapsed_full:>14.2f}초')
print(f'{"요청당 비용":<16} ${c_mini:>16.6f} ${c_full:>14.6f}')
print(f'{f"월 {MY_MONTHLY_TARGET:,}건 절약액":<14} ${saving:>18.2f} = {saving * WON:>8,.0f}원 절약')
print()
print('💬 슬랙 #day5-심화 채널에 아래 내용을 공유하세요:')
print()
print('  우리 팀이 선택할 모델: gpt-4o-mini / gpt-4o  (택1)')
print('  이유: (응답 품질 차이가 우리 서비스에서 용납 가능한가?')
print('         비용 절약($)이 품질 차이보다 중요한가?)')
print('    →')

---
## 📬 제출 & 자가 체크

### ✅ 자가 체크리스트

**[요소 4 · 상태 관리]**
- [ ] Step 1-①: LLM이 이전 대화를 기억하지 못하는 것을 직접 확인했다
- [ ] Step 1-②: `MessagesPlaceholder`로 이전 대화를 주입해 기억이 복원되는 것을 확인했다
- [ ] Step 1-③: `RunnableWithMessageHistory`로 자동 메모리가 동작하는 것을 확인했다
- [ ] 🔰 기본 미션: Memory 전략 선택 출력을 `day5_실습_팀설계서v1.md` 섹션 4에 붙여넣었다

**[요소 6 · 비용 계획]**
- [ ] Step 1-②: 내 서비스 프롬프트를 tiktoken으로 실측했다
- [ ] Step 2-②: 내 서비스의 요청당 비용($)을 말할 수 있다
- [ ] Step 3-②: 내 서비스의 월 목표 요청 수와 월 비용을 말할 수 있다
- [ ] 🔰 기본 미션: 비용 출력을 `day5_실습_팀설계서v1.md` 섹션 7에 붙여넣었다
- [ ] (⭐ 심화 C) gpt-4o vs gpt-4o-mini 모델 선택 이유를 슬랙 **#day5-심화**에 공유했다

### 📤 제출 형식

**상태 관리 기본 미션** → `day5_실습_팀설계서v1.md` 섹션 4 직접 기입  
**비용 계획 기본 미션** → `day5_실습_팀설계서v1.md` 섹션 7 직접 기입  
→ 완성된 설계서 파일을 슬랙 **#day5-팀설계서** 에 업로드

```
[제출 형식 예시]
팀명: ___  서비스명: ___
Memory 전략: Sliding Window / 이유: 짧은 Q&A 위주
요청당 비용: $0.000xxx  월 N,000건: $x.xx
```

---
### ➡️ 다음 연결

오늘로 LangChain 기초 5일 과정을 마칩니다. 다음 수업일은 **9/14(월)** — 오늘 설계한 서비스를
실제로 운영 가능한 형태로 확장하는 파이프라인 모듈이 시작됩니다.

그리고 10월 중에는 홍경수 강사님의 RAG(검색 증강 생성) 트랙에서, 오늘 설계서에 적어둔 서비스에
**문서 기반 검색·답변 기능**을 추가하게 됩니다 — 지금 만든 설계도가 그때 다시 쓰입니다.
(정확한 날짜는 홍경수 강사 세부계획 확정 후 별도 공지)

키워드 미리 보기(파이프라인 모듈): `async def` / `await ainvoke()` / `asyncio.gather()` / `Semaphore`